In [5]:
from equiv_dens.scripts import transform_orbitals
import numpy as np
import pyscf
from equiv_dens.data.hamiltonian_dataset import HamiltonianDataset

In [43]:
pyscf_orb = np.load('datasets/h2o_dynamic_pyscf_def2svp_dft_f.npy', allow_pickle=True)
svp_orb = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
svp_orb_db = HamiltonianDataset('datasets/h2o_pbe-def2svp_4999.db')


In [44]:
#compare svp npy vs db hamiltonians

for i in range(len(svp_orb['full_hamiltonian'][0, 0])):
    print('i', i)
    print('npy ham', svp_orb['full_hamiltonian'][0, i])
    print('db ham', svp_orb_db.collate_fn([0])['full_hamiltonian'][0, i])

i 0
npy ham [-1.8720873e+01  6.8529019e+00  3.2023270e+00  8.7839998e-03
 -1.4163000e-02 -4.4080000e-03  1.1830000e-03 -1.7930000e-03
 -5.0000002e-04  1.1180000e-03 -8.4100000e-04 -2.0400000e-04
  4.5100000e-04 -4.5399999e-04  7.4718797e-01  1.1833390e+00
  3.4163201e-01  8.0897301e-01  9.4573200e-01  1.4141010e+00
  1.4073480e+00 -2.3995631e+00  1.4270260e+00 -8.0216199e-01]
db ham tensor([-1.8721e+01,  6.8529e+00,  3.2023e+00,  8.7840e-03, -1.4163e-02,
        -4.4080e-03,  1.1830e-03, -1.7930e-03, -5.0000e-04,  1.1180e-03,
        -8.4100e-04, -2.0400e-04,  4.5100e-04, -4.5400e-04,  7.4719e-01,
         1.1833e+00,  3.4163e-01,  8.0897e-01,  9.4573e-01,  1.4141e+00,
         1.4073e+00, -2.3996e+00,  1.4270e+00, -8.0216e-01])
i 1
npy ham [ 6.852902 -3.108961 -1.930825 -0.073932  0.095154  0.017338 -0.055106
  0.075039  0.016304 -0.058327  0.052932  0.010606 -0.017949  0.030493
 -0.653431 -0.88072  -0.221698 -0.498094 -0.59164  -1.065698 -1.031457
  1.159521 -0.677284  0.397717]
db h

In [45]:
import scipy as sp
def orbitals_from_hamiltonian(hamiltonians, overlaps):
    orbital_coeffs = []
    for i in range(hamiltonians.shape[0]):
        en, coefs = sp.linalg.eigh(a=hamiltonians[i], b=overlaps[i])
        occ_orbitals = en > 0
        orbital_coeffs.append(coefs)
    return orbital_coeffs

def parse_orbitals(orbitals, atom_types, basis_def):
    split_orbitals = []
    count = 0
    for at in atom_types:
        basis_at = basis_def[at]
        for orb in basis_at:
            L = orb[2]
            split_orbitals.append(orbitals[:, count:(count + (2 * L) + 1)])
            count += (2 * L) + 1
    return split_orbitals

In [46]:
hamiltonians_svp = svp_orb['full_hamiltonian']
overlap_svp = svp_orb['overlap_matrix']

orbitals_svp = orbitals_from_hamiltonian(hamiltonians_svp, overlap_svp)

In [47]:
print(orbitals_svp[0][[0], :])
print(pyscf_orb[0][1]['mo_coeff'][[0], :])
atom_types = svp_orb['atom_types']
print('')
basis_def = np.load('datasets/631gss_orbital_basis_df.npy', allow_pickle=True).item()
split_svp = parse_orbitals(orbitals_svp[0][[0], :], atom_types, basis_def)
split_pyscf = parse_orbitals(pyscf_orb[0][1]['mo_coeff'][[0], :], atom_types, basis_def)
for i in range(len(split_svp)):
    print('svp', split_svp[i])
    print('pyscf', split_pyscf[i])
    print('')

[[ 9.8843747e-01  2.7187237e-01  2.5951635e-02  1.2193479e-01
  -5.1288208e-07 -9.7053431e-02  3.3242863e-02  4.1871078e-02
   9.4696596e-02 -2.1718806e-04  1.6276111e-01  8.6943425e-02
  -2.1250114e-01 -9.8385744e-02  1.5627623e-06  2.4220919e-06
  -2.7761315e-03 -8.3137155e-02 -1.1965891e-01  6.6252193e-07
   2.3911716e-06 -2.1731257e-02 -6.4714178e-02  6.9363251e-02]]
[[ 9.94808341e-01 -2.07392181e-01  1.94493088e-02 -9.15945360e-02
  -6.48337904e-09  7.98352206e-02  3.56905161e-02 -5.83853757e-03
   5.80197684e-03  2.18694176e-08 -7.08572078e-02 -3.81354889e-02
   4.83547503e-02  1.13709047e-02  6.96765697e-08  3.19184232e-08
   1.17821378e-02 -3.53745918e-03 -3.02240913e-07  3.07438075e-02
   1.32350856e-07  5.04243835e-03 -7.71707365e-02  3.21794835e-02]]

svp [[0.9884375]]
pyscf [[0.99480834]]

svp [[0.27187237]]
pyscf [[-0.20739218]]

svp [[0.02595164]]
pyscf [[0.01944931]]

svp [[ 1.2193479e-01 -5.1288208e-07 -9.7053431e-02]]
pyscf [[-9.15945360e-02 -6.48337904e-09  7.98352206

In [53]:
import copy
from equiv_dens.scripts.transform_hamiltonians import transform
%load_ext autoreload
%autoreload 2

svp_orb_201 = copy.deepcopy(svp_orb)
svp_orb_210 = copy.deepcopy(svp_orb)


svp_orb_201['full_hamiltonian'] = transform(svp_orb_201['full_hamiltonian'], atom_types, 'def2-SVP_to_pyscf_201')
svp_orb_210['full_hamiltonian'] = transform(svp_orb_210['full_hamiltonian'], atom_types, 'def2-SVP_to_pyscf_210')
svp_orb_201['overlap_matrix'] = transform(svp_orb_201['overlap_matrix'], atom_types, 'def2-SVP_to_pyscf_201')
svp_orb_210['overlap_matrix'] = transform(svp_orb_210['overlap_matrix'], atom_types, 'def2-SVP_to_pyscf_210')
    
np.save('datasets/h2o_dynamic_ham_def2svp_201_dft_f.npy', svp_orb_201, allow_pickle=True)
np.save('datasets/h2o_dynamic_ham_def2svp_210_dft_f.npy', svp_orb_210, allow_pickle=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
atoms ['O' 'H' 'H']
svr aroms to orbs sssppd
svr aroms to orbs ssp
svr aroms to orbs ssp
orbitals sssppdsspssp
orbitals order [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
transform_indices [array([0]), array([1]), array([2]), array([5, 3, 4]), array([8, 6, 7]), array([ 9, 10, 11, 12, 13]), array([14]), array([15]), array([18, 16, 17]), array([19]), array([20]), array([23, 21, 22])]
atoms ['O' 'H' 'H']
svr aroms to orbs sssppd
svr aroms to orbs ssp
svr aroms to orbs ssp
orbitals sssppdsspssp
orbitals order [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
transform_indices [array([0]), array([1]), array([2]), array([5, 4, 3]), array([8, 7, 6]), array([ 9, 10, 11, 12, 13]), array([14]), array([15]), array([18, 17, 16]), array([19]), array([20]), array([23, 22, 21])]
atoms ['O' 'H' 'H']
svr aroms to orbs sssppd
svr aroms to orbs ssp
svr aroms to orbs ssp
orbitals sssppdsspssp
orbitals order [0, 1, 2, 3, 4, 5, 6, 7, 8, 